# Phase 4: Synthetic Model Training & Utility Evaluation

This notebook:
1. Executes the synthetic downstream training script `scripts/train_synthetic.py` (which trains models on synthetic splits and tests on real splits).
2. Compiles classification performance metrics (ROC-AUC, F1-Score, Precision, Recall) across all 5 seeds for both datasets and both generator types (CTGAN and TVAE).
3. Displays a final performance comparison table comparing Real baseline vs CTGAN synthetic vs TVAE synthetic.

In [1]:
import os
import sys
import json
import pandas as pd
import numpy as np

## 1. Run Downstream Training Script

We run the downstream synthetic-trained model script if predictions are missing.

In [2]:
ctgan_summary_file = "../../results/ctgan/german_credit_ctgan_xgboost_summary.json"
if not os.path.exists(ctgan_summary_file):
    print("Running synthetic model training script (python scripts/train_synthetic.py)...")
    import subprocess
    subprocess.run([sys.executable, "../../scripts/train_synthetic.py"], check=True)
else:
    print("Synthetic training summaries already exist. Skipping script execution.")

Synthetic training summaries already exist. Skipping script execution.


## 2. Load and Compare Metrics

We load Real baseline, CTGAN synthetic, and TVAE synthetic results for both German Credit and GMSC, calculate metrics `Mean ± Std`, and render the final comparison table.

In [3]:
def get_aggregated_stats(summary_path):
    with open(summary_path, 'r') as f:
        summary = json.load(f)
    
    runs = summary["runs"]
    df = pd.DataFrame(runs)
    
    stats = {}
    for col, metric_name in [('roc_auc', 'ROC-AUC'), ('f1', 'F1-Score'), ('precision', 'Precision'), ('recall', 'Recall')]:
        mean_val = df[col].mean()
        std_val = df[col].std()
        stats[metric_name] = f"{mean_val:.4f} ± {std_val:.4f}"
        
    return stats

def build_comparison_table(dataset_name):
    real_path = f"../../results/baseline/{dataset_name}_real_xgboost_summary.json"
    ctgan_path = f"../../results/ctgan/{dataset_name}_ctgan_xgboost_summary.json"
    tvae_path = f"../../results/tvae/{dataset_name}_tvae_xgboost_summary.json"
    
    rows = []
    for name, path in [("Real Baseline", real_path), ("CTGAN Synthetic", ctgan_path), ("TVAE Synthetic", tvae_path)]:
        if os.path.exists(path):
            stats = get_aggregated_stats(path)
            stats["Model Type"] = name
            rows.append(stats)
            
    df_comp = pd.DataFrame(rows)[["Model Type", "ROC-AUC", "F1-Score", "Precision", "Recall"]]
    print(f"=== {dataset_name.upper()} UTILITY COMPARISON ===")
    display(df_comp)
    print("\n" + "="*50 + "\n")
    return df_comp

german_comparison = build_comparison_table("german_credit")
gmsc_comparison = build_comparison_table("gmsc")

=== GERMAN_CREDIT UTILITY COMPARISON ===


,Model Type,ROC-AUC,F1-Score,Precision,Recall
0,Real Baseline,0.7715 ± 0.0192,0.5825 ± 0.0226,0.5341 ± 0.0357,0.6422 ± 0.0241
1,CTGAN Synthetic,0.4946 ± 0.0372,0.3145 ± 0.0615,0.3042 ± 0.0502,0.3311 ± 0.0851
2,TVAE Synthetic,0.6946 ± 0.0203,0.3676 ± 0.1067,0.5313 ± 0.0840,0.3111 ± 0.1503




=== GMSC UTILITY COMPARISON ===


,Model Type,ROC-AUC,F1-Score,Precision,Recall
0,Real Baseline,0.8362 ± 0.0195,0.3105 ± 0.0164,0.1965 ± 0.0106,0.7400 ± 0.0414
1,CTGAN Synthetic,0.7778 ± 0.0379,0.3067 ± 0.0465,0.2173 ± 0.0584,0.5690 ± 0.0843
2,TVAE Synthetic,0.7853 ± 0.0364,0.3334 ± 0.0504,0.2844 ± 0.0443,0.4160 ± 0.1017
